# '''

# This notebook is your working file for Assignment 2.  
# All analysis must be done here. Ensure that all code runs from top to bottom without errors.

# '''

In [ ]:
# Section 0 – Setup


import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# Optional: statsmodels for OLS
import statsmodels.api as sm

# Optional: ML models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional: optimisation
# !pip install pulp  # uncomment if needed
import pulp

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
# Load data

# Update the path if needed
df = pd.read_csv(-------) #see earlier lab sessions

In [ ]:
df.head()


## Section 1 – Descriptive Analytics

# Tasks:
# - Produce summary statistics for key variables
# - Visualise distributions of important variables
# - Compare responders vs non-responders
# - Highlight at least two insights about how responders differ from non-responders

# Use both tables and plots, and write a short narrative.
# Summary statistics for numeric variables

In [ ]:
df.describe()
# Response rate

df["Response"].value_counts()
# Example: distribution of AvgBasketValue by Response

In [ ]:
sns.boxplot(data=df, x="Response", y="AvgBasketValue")
plt.title("AvgBasketValue by Response")
plt.show()

# Example: distribution of VisitsLast3M by Response

sns.boxplot(data=df, x="Response", y="VisitsLast3M")
plt.title("VisitsLast3M by Response")
plt.show()

# Example: segment distribution by Response

sns.countplot(data=df, x="Segment", hue="Response")
plt.title("Segment vs Response")
plt.show()

## Section 2 – OLS Regression for Customer Value

# Tasks:
# - Choose a continuous measure of customer value (e.g., AvgBasketValue or a constructed metric)
# - Justify your choice
# - Build an OLS regression model
# - Interpret coefficients and discuss which customers are more valuable
# - Comment on model adequacy (R², residuals, multicollinearity if checked)

# Example: using AvgBasketValue as the dependent variable

In [ ]:
value_var = "AvgBasketValue"

# Choose predictors (you may adjust this list)
predictors = [
    "Age",
    "Income",
    "TenureMonths",
    "OnlineShare",
    "VisitsLast3M",
    "PastPromoResponse"
]

# Handle categorical variables (e.g., Segment)
df_model = df.copy()
df_model = pd.get_dummies(df_model, columns=["Segment"], drop_first=True)

# Update predictors if including segment dummies
predictors_extended = predictors + [col for col in df_model.columns if col.startswith("Segment_")]

In [ ]:

X = df_model[predictors_extended]
y = df_model[value_var]

# Add constant
X = X.astype(float)
X_const = sm.add_constant(X)


X_const = sm.add_constant(X)

ols_model = sm.OLS(y, X_const).fit()

In [ ]:
print(ols_model.summary())


In [ ]:
# Residual plot

residuals = ols_model.resid
fitted = ols_model.fittedvalues

plt.scatter(fitted, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.show()


## Section 3 – Predictive Modelling (Response)

# Tasks:
# - Build two models to predict Response:
#   - Logistic regression
#   - One machine-learning model (e.g., Random Forest)
# - Use a train/test split
# - Evaluate using accuracy, precision, recall, ROC-AUC (and any other relevant metrics)
# - Compare models and choose one for promotion decisions
# Prepare data for classification



In [ ]:
target = "Response"

# Reuse df_model with dummies
X_cls = df_model.drop(columns=[target, value_var])  # drop target and value metric
y_cls = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

X_train.shape, X_test.shape


In [ ]:
# rescaling if needed

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


In [ ]:
# Logistic Regression

log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(X_train_scaled, y_train)

y_pred_log = log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]


In [ ]:
# Random Forest (example ML model)

rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    
    print(f"=== {model_name} ===")
    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"ROC-AUC:   {auc:.3f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("-" * 40)

evaluate_model(y_test, y_pred_log, y_prob_log, model_name="Logistic Regression")
evaluate_model(y_test, y_pred_rf, y_prob_rf, model_name="Random Forest")

In [ ]:
# ROC curves

fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.plot(fpr_log, tpr_log, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()


## Section 4 – Prescriptive Analytics (Optimisation)

# ------------------------------------------------------------
# Section 4 – Prescriptive Analytics (Optimisation)
# ------------------------------------------------------------
# In this section, we decide WHICH customers to target in the next promotion.
# We use:
#   - Predicted probability of responding (from our chosen model)
#   - Expected margin if they respond
#   - PromoCost for sending the promotion
#
# Our goal:
#   Maximise TOTAL expected profit, subject to a fixed promotion budget.
# ------------------------------------------------------------

# ------------------------------------------------------------
# 4.0 Choose the predictive model for probabilities
# ------------------------------------------------------------
# We use the Random Forest model here, but you may choose logistic regression instead.

In [ ]:
chosen_model = rf_clf

# Predict response probability for ALL customers
prob_response = chosen_model.predict_proba(X_cls)[:, 1]

df_opt = df.copy()
df_opt["PredictedProbResponse"] = prob_response

df_opt[["CustomerID", "PredictedProbResponse"]].head()


# ------------------------------------------------------------
# 4.1 Compute Expected Profit for each customer
# ------------------------------------------------------------
# Formula:
#   ExpectedProfit_i = p_i * ExpectedMarginIfRespond_i - PromoCost_i

In [ ]:
df_opt["ExpectedProfit"] = (
    df_opt["PredictedProbResponse"] * df_opt["ExpectedMarginIfRespond"]
    - df_opt["PromoCost"]
)

df_opt[["CustomerID", "PredictedProbResponse", "ExpectedMarginIfRespond",
        "PromoCost", "ExpectedProfit"]].head()


# ------------------------------------------------------------
# 4.2 Define the promotion budget
# ------------------------------------------------------------
# You may choose and justify a different value in your report.

In [ ]:

budget = -------- # what amoutn do you want to choose?
total_promo_cost = df_opt["PromoCost"].sum()
total_promo_cost, budget


# ------------------------------------------------------------
# 4.3 Set up the optimisation model in PuLP
# ------------------------------------------------------------

In [ ]:
customers = df_opt.index.tolist()

# Decision variable:
#   x[i] = 1 if customer i is targeted, 0 otherwise
x = pulp.LpVariable.dicts("x", customers, lowBound=0, upBound=1, cat="Binary")

# Create the optimisation problem
problem = pulp.LpProblem("NovaMart_Promotion_Optimisation", ---------------) # is it pulp.LpMaximize or pulp.LpMinimize ???

# ------------------------------------------------------------
# 4.4 Objective function (Maximise expected profit)
# ------------------------------------------------------------
# This mirrors the lab structure:
#   12*Q + 10*R + 14*S
# but now with many more terms.

In [ ]:
objective_expression = sum(
    x[i] * df_opt.loc[i, "ExpectedProfit"]
    for i in customers
)

problem += objective_expression


# ------------------------------------------------------------
# 4.5 Budget constraint (LAB STYLE)
# ------------------------------------------------------------
# In the lab, constraints looked like:
#   12*Q + 10*R + 14*S <= 1200
#
# Here we do the SAME THING:
#   SUM_i [ x[i] * PromoCost_i ] <= budget
#
# Below is a single linear expression, just like the lab examples.

In [ ]:
promo_cost_constraint = sum(
    x[i] * df_opt.loc[i, "PromoCost"]
    for i in customers
)

problem += promo_cost_constraint <= budget

# ------------------------------------------------------------
# 4.6 Solve the optimisation problem
# ------------------------------------------------------------

In [ ]:
problem.solve(pulp.PULP_CBC_CMD(msg=False))

print("Solver status:", pulp.LpStatus[problem.status])

# ------------------------------------------------------------
# 4.7 Extract selected customers
# ------------------------------------------------------------

In [ ]:

df_opt["Target"] = [int(x[i].value()) for i in customers]

df_opt["Target"].value_counts()

